# 根据Index把LLM生成的正边和confirmed-negative拼接起来

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

def generate_fixed_pos_ratio_datasets(
    pos_file_path, 
    neg_file_path, 
    seed=42
):
    # 设置随机种子
    np.random.seed(seed)
    
    print(f"===== 1. 数据加载与预处理 =====")
    
    # ---------------------------
    # 1. 加载负样本 (作为基准池)
    # ---------------------------
    print(f"读取负样本: {neg_file_path}")
    df_neg_pool = pd.read_csv(neg_file_path)
    
    # 标准化列名
    if 'src' not in df_neg_pool.columns: # 假设只有两列
        df_neg_pool = df_neg_pool.iloc[:, :2]
        df_neg_pool.columns = ['src', 'dst']
        
    df_neg_pool = df_neg_pool[['src', 'dst']]
    df_neg_pool['src'] = df_neg_pool['src'].astype(int)
    df_neg_pool['dst'] = df_neg_pool['dst'].astype(int)
    df_neg_pool['label'] = 2  # 负样本 Label
    
    total_neg_available = len(df_neg_pool)
    print(f"-> 原始负样本池大小: {total_neg_available}")

    # ---------------------------
    # 2. 加载正样本 (作为候选池)
    # ---------------------------
    print(f"读取正样本: {pos_file_path}")
    df_pos_all = pd.read_csv(pos_file_path)
    
    # 筛选 type=positive 并重命名列
    df_pos_cand = df_pos_all[df_pos_all['type'] == 'positive'].copy()
    df_pos_cand = df_pos_cand.rename(columns={'index_A': 'src', 'index_B': 'dst'})
    df_pos_cand = df_pos_cand[['src', 'dst']]
    df_pos_cand['src'] = df_pos_cand['src'].astype(int)
    df_pos_cand['dst'] = df_pos_cand['dst'].astype(int)
    df_pos_cand['label'] = 1 # 正样本 Label
    
    print(f"-> 原始正样本候选数量: {len(df_pos_cand)}")

    # ---------------------------
    # 3. 冲突清洗 (关键步骤)
    # ---------------------------
    # 既然负样本池是基准，如果正样本里有跟负样本重复的边，必须剔除正样本
    print("正在清洗正样本 (剔除与负样本池冲突的边)...")
    
    # 构建负样本查找集 (处理无向边)
    neg_edge_set = set()
    for u, v in zip(df_neg_pool['src'], df_neg_pool['dst']):
        neg_edge_set.add(tuple(sorted((u, v))))
        
    # 筛选不冲突的正样本
    # 使用列表推导式比 iterrows 快
    keep_mask = [tuple(sorted((u, v))) not in neg_edge_set for u, v in zip(df_pos_cand['src'], df_pos_cand['dst'])]
    pos_pool = df_pos_cand[keep_mask].copy()
    
    print(f"-> 冲突剔除数量: {len(df_pos_cand) - len(pos_pool)}")
    print(f"-> 最终可用正样本池: {len(pos_pool)}")

    # ==========================================
    # 4. 核心逻辑实现 (按照你提供的思路)
    # ==========================================



    # 保存汇总信息到 CSV 方便查阅
    # summary_df.to_csv(save_dir / "dataset_summary.csv", index=False)

# ==========================
# 运行
# ==========================
pos_file = '/home/yyyy/codework/GARplus/DiGress/DiGress-main/data/PPI/raw/protein_protein_with_type.csv'
neg_file = '/home/yyyy/codework/GARplus/GNN/code/data/neg_protein_protein.csv' # 确保此文件存在
#TODO 构建edges.csv和node.csv文件
generate_fixed_pos_ratio_datasets(pos_file, neg_file)

===== 1. 数据加载与预处理 =====
读取负样本: /home/yyyy/codework/GARplus/GNN/code/data/neg_protein_protein.csv
-> 原始负样本池大小: 1426
读取正样本: /home/yyyy/codework/GARplus/DiGress/DiGress-main/data/PPI/raw/protein_protein_with_type.csv
-> 原始正样本候选数量: 681684
正在清洗正样本 (剔除与负样本池冲突的边)...
-> 冲突剔除数量: 292
-> 最终可用正样本池: 681392

===== 数据集生成配置 =====
原始负边池大小 (total_neg_available): 1426
设定固定正边数量 (fixed_pos_num): 14260
-> 已锁定正样本集，数量: 14260
[生成完成] Ratio 10x | Pos: 14260 | Neg: 1426 | Total: 15686
[生成完成] Ratio 20x | Pos: 14260 | Neg: 713 | Total: 14973
[生成完成] Ratio 30x | Pos: 14260 | Neg: 475 | Total: 14735
[生成完成] Ratio 40x | Pos: 14260 | Neg: 356 | Total: 14616
[生成完成] Ratio 50x | Pos: 14260 | Neg: 285 | Total: 14545
[生成完成] Ratio 60x | Pos: 14260 | Neg: 237 | Total: 14497

===== 数据集统计汇总 =====
  ratio_label imbalance_ratio  pos_count  neg_count neg_fraction  \
0         10x            10:1      14260       1426      100.00%   
1         20x            20:1      14260        713       50.00%   
2         30x            30:1    

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

def generate_outputs_with_pos_neg_swap_fallback(
    pos_file_path,
    neg_file_path,
    output_dir,
    seed=42,
    print_top_missed=50,   # 打印前多少条没命中的负边
    save_missed_csv=True,  # 是否保存没命中的负边到csv
):
    np.random.seed(seed)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print("===== 1) 读取负样本池 =====")
    df_neg_pool = pd.read_csv(neg_file_path)

    if 'src' not in df_neg_pool.columns or 'dst' not in df_neg_pool.columns:
        df_neg_pool = df_neg_pool.iloc[:, :2].copy()
        df_neg_pool.columns = ['src', 'dst']
    df_neg_pool = df_neg_pool[['src', 'dst']].copy()
    df_neg_pool['src'] = df_neg_pool['src'].astype(int)
    df_neg_pool['dst'] = df_neg_pool['dst'].astype(int)
    print(f"-> 负样本数量: {len(df_neg_pool)}")

    neg_directed_set = set(zip(df_neg_pool['src'], df_neg_pool['dst']))
    neg_reversed_set = set(zip(df_neg_pool['dst'], df_neg_pool['src']))

    print("===== 2) 读取正样本底稿文件 =====")
    df_pos_all = pd.read_csv(pos_file_path)

    if 'index_A' not in df_pos_all.columns or 'index_B' not in df_pos_all.columns:
        raise ValueError("pos_file 里必须包含 index_A 和 index_B 列")
    if 'type' not in df_pos_all.columns:
        raise ValueError("pos_file 里必须包含 type 列，且正样本候选行的 type == 'positive'")

    df_pos_all['index_A'] = df_pos_all['index_A'].astype(int)
    df_pos_all['index_B'] = df_pos_all['index_B'].astype(int)

    df_out = df_pos_all.copy()
    df_out['type'] = ""

    print("===== 3) 标注可用正样本（剔除与负样本冲突：同向或反向） =====")
    pos_cand_mask = (df_pos_all['type'] == 'positive')
    pos_cand = df_pos_all.loc[pos_cand_mask, ['index_A', 'index_B']].copy()

    pos_pairs = list(zip(pos_cand['index_A'], pos_cand['index_B']))
    keep_mask = [ (p not in neg_directed_set) and (p not in neg_reversed_set) for p in pos_pairs ]

    num_pos_cand = len(pos_cand)
    num_pos_keep = int(np.sum(keep_mask))
    print(f"-> 原始正样本候选: {num_pos_cand}")
    print(f"-> 与负样本冲突剔除: {num_pos_cand - num_pos_keep}")
    print(f"-> 最终可用正样本: {num_pos_keep}")

    keep_indices = pos_cand.index.to_numpy()[np.array(keep_mask, dtype=bool)]
    df_out.loc[keep_indices, 'type'] = 'positive'

    print("===== 4) 负样本标注：先( src,dst )匹配；没匹配到再交换( dst,src )匹配 =====")
    edge_to_rows = {}
    for i, (a, b) in enumerate(zip(df_out['index_A'], df_out['index_B'])):
        edge_to_rows.setdefault((a, b), []).append(i)

    updated_existing = 0
    updated_by_swap = 0
    appended_new = 0
    conflict_pos_neg = 0
    new_rows = []

    # ✅ 收集“两次都没命中”的负边
    missed_neg_edges = []  # 每条记录: (src, dst, swapped_src, swapped_dst)

    for u, v in zip(df_neg_pool['src'], df_neg_pool['dst']):
        key1 = (int(u), int(v))
        key2 = (int(v), int(u))

        matched = False

        if key1 in edge_to_rows:
            for row_i in edge_to_rows[key1]:
                if df_out.at[row_i, 'type'] == 'positive':
                    conflict_pos_neg += 1
                    continue
                df_out.at[row_i, 'type'] = 'negative'
                updated_existing += 1
            matched = True

        if (not matched) and (key2 in edge_to_rows):
            for row_i in edge_to_rows[key2]:
                if df_out.at[row_i, 'type'] == 'positive':
                    conflict_pos_neg += 1
                    continue
                df_out.at[row_i, 'type'] = 'negative'
                updated_by_swap += 1
            matched = True

        if not matched:
            missed_neg_edges.append((key1[0], key1[1], key2[0], key2[1]))

            row_dict = {col: np.nan for col in df_out.columns}
            row_dict['index_A'] = key1[0]
            row_dict['index_B'] = key1[1]
            row_dict['type'] = 'negative'
            new_rows.append(row_dict)
            appended_new += 1

    if new_rows:
        df_out = pd.concat([df_out, pd.DataFrame(new_rows)], ignore_index=True)

    print(f"-> 负样本同向命中并标注次数: {updated_existing}")
    print(f"-> 负样本交换后命中并标注次数: {updated_by_swap}")
    print(f"-> 底稿缺失而追加的负样本边数: {appended_new}")
    if conflict_pos_neg > 0:
        print(f"[警告] 发现 {conflict_pos_neg} 个负边与已标 positive 的行冲突（已跳过覆盖）")

    # ✅ 打印/保存没命中的负边
    print(f"-> 两次都没命中的负边数量: {len(missed_neg_edges)}")
    if len(missed_neg_edges) > 0:
        topk = min(print_top_missed, len(missed_neg_edges))
        print(f"   展示前 {topk} 条（src,dst 以及交换后 dst,src）：")
        for i in range(topk):
            s, d, ss, dd = missed_neg_edges[i]
            print(f"   #{i+1}: ({s}, {d})  | swap -> ({ss}, {dd})")

        if save_missed_csv:
            missed_path = output_dir / "missed_negative_edges.csv"
            df_missed = pd.DataFrame(
                missed_neg_edges,
                columns=["src", "dst", "swap_src", "swap_dst"]
            )
            df_missed.to_csv(missed_path, index=False)
            print(f"✅ 已保存没命中的负边到: {missed_path}")

    print("===== 5) 写出两个文件 =====")
    out_full = output_dir / "ppi_with_pos_neg_full.csv"
    out_gnn  = output_dir / "ppi_for_gnn_edges.csv"

    df_out.to_csv(out_full, index=False)

    df_gnn = df_out[['index_A', 'index_B', 'type']].copy()
    df_gnn = df_gnn.rename(columns={'index_A': 'src', 'index_B': 'dst'})
    df_gnn.to_csv(out_gnn, index=False)

    print(f"✅ 输出文件1: {out_full}")
    print(f"✅ 输出文件2: {out_gnn}")

    return out_full, out_gnn


# 运行示例
pos_file = '/home/yyyy/codework/GARplus/DiGress/DiGress-main/data/PPI/raw/protein_protein_with_type.csv'
neg_file = '/home/yyyy/codework/GARplus/GNN/code/data/neg_protein_protein.csv'
output_dir = '/home/yyyy/codework/GARplus/DiGress/DiGress-main/data/PPI/raw'

generate_outputs_with_pos_neg_swap_fallback(pos_file, neg_file, output_dir)


===== 1) 读取负样本池 =====
-> 负样本数量: 1426
===== 2) 读取正样本底稿文件 =====
===== 3) 标注可用正样本（剔除与负样本冲突：同向或反向） =====
-> 原始正样本候选: 681684
-> 与负样本冲突剔除: 292
-> 最终可用正样本: 681392
===== 4) 负样本标注：先( src,dst )匹配；没匹配到再交换( dst,src )匹配 =====
-> 负样本同向命中并标注次数: 164
-> 负样本交换后命中并标注次数: 150
-> 底稿缺失而追加的负样本边数: 1112
-> 两次都没命中的负边数量: 1112
   展示前 50 条（src,dst 以及交换后 dst,src）：
   #1: (177418, 198682)  | swap -> (198682, 177418)
   #2: (194806, 163592)  | swap -> (163592, 194806)
   #3: (158677, 187470)  | swap -> (187470, 158677)
   #4: (194595, 156340)  | swap -> (156340, 194595)
   #5: (189245, 154490)  | swap -> (154490, 189245)
   #6: (159813, 195748)  | swap -> (195748, 159813)
   #7: (179034, 154834)  | swap -> (154834, 179034)
   #8: (156390, 158304)  | swap -> (158304, 156390)
   #9: (181378, 179642)  | swap -> (179642, 181378)
   #10: (154789, 166167)  | swap -> (166167, 154789)
   #11: (154789, 166167)  | swap -> (166167, 154789)
   #12: (185530, 166167)  | swap -> (166167, 185530)
   #13: (198408, 166167)  | swap -> (

(PosixPath('/home/yyyy/codework/GARplus/DiGress/DiGress-main/data/PPI/raw/ppi_with_pos_neg_full.csv'),
 PosixPath('/home/yyyy/codework/GARplus/DiGress/DiGress-main/data/PPI/raw/ppi_for_gnn_edges.csv'))

# similarity score computation

In [4]:
import os
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# 0) 配置：按你的真实列名写好的
# =========================
PROTEIN_CSV = "/home/yyyy/codework/GARplus/DiGress/DiGress-main/data/PPI/raw/protein.csv"
PPI_CSV = "/home/yyyy/codework/GARplus/DiGress/DiGress-main/data/PPI/raw/protein_protein_with_type.csv"

# protein 表
PROT_ID_COL = "biogrid_id"     # 用这个和边表对齐
TEXT_COLS = [
    "location",
    "pathway",
    "domain",
    "entryname",
    "synonyms",
    "protein_names",
    "Protein families",
    "Sequence similarities",
]

# edge 表
EDGE_U_COL = "BioGRID ID Interactor A"
EDGE_V_COL = "BioGRID ID Interactor B"

# 输出列名
OUT_SCORE_COL = "ml_score"
OUT_PRED_COL = "ml_pred"       # 0/1 predicate
THRESHOLD = 0.5                # 你可以调


# =========================
# 1) 工具函数
# =========================
def clean_id(x):
    """把 119816.0 / '119816' / '119816.abc' 都统一成 '119816' """
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return None
    # 去掉小数点后 / 或者 split('.') 的情况
    s = s.split(".")[0]
    # 只保留数字（保险做法）
    m = re.match(r"^\d+$", s)
    return s if m else None


def norm_text(x):
    """文本清洗：None->''，小写，去掉奇怪符号，压缩空格"""
    if pd.isna(x):
        return ""
    s = str(x).lower()
    s = re.sub(r"[^a-z0-9\s;:_\-\(\)\[\]\./]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def build_protein_text_row(row: pd.Series) -> str:
    """把一行 protein 的多个字段拼成一个 '文档' """
    parts = []
    for c in TEXT_COLS:
        if c in row:
            parts.append(norm_text(row[c]))
    return " ".join([p for p in parts if p])


# =========================
# 2) 主流程
# =========================
def main():
    prot = pd.read_csv(PROTEIN_CSV, low_memory=False)
    ppi = pd.read_csv(PPI_CSV, low_memory=False)

    # ---- A. 清洗 protein id，并建立 protein_id -> 文本文档
    if PROT_ID_COL not in prot.columns:
        raise ValueError(f"protein.csv 里找不到列 {PROT_ID_COL}")

    prot[PROT_ID_COL] = prot[PROT_ID_COL].apply(clean_id)
    prot = prot.dropna(subset=[PROT_ID_COL])

    # 建立 protein 文本
    prot_text = {}
    for _, row in prot.iterrows():
        pid = row[PROT_ID_COL]
        prot_text[pid] = build_protein_text_row(row)

    prot_ids = list(prot_text.keys())
    corpus = [prot_text[pid] for pid in prot_ids]

    # ---- B. TF-IDF 向量化（轻量且好用）
    vectorizer = TfidfVectorizer(
        min_df=2,
        ngram_range=(1, 2),
        max_features=200000,
    )
    X = vectorizer.fit_transform(corpus)  # [num_proteins, vocab]
    pid2idx = {pid: i for i, pid in enumerate(prot_ids)}

    # ---- C. 清洗 edge 端点 id
    if EDGE_U_COL not in ppi.columns or EDGE_V_COL not in ppi.columns:
        raise ValueError(f"edge csv 里找不到 {EDGE_U_COL} 或 {EDGE_V_COL}")

    us = ppi[EDGE_U_COL].apply(clean_id).tolist()
    vs = ppi[EDGE_V_COL].apply(clean_id).tolist()

    # ---- D. 对每条边算 cosine similarity
    scores = []
    missing = 0

    for u, v in tqdm(zip(us, vs), total=len(us), desc="Scoring edges"):
        if u is None or v is None:
            scores.append(0.0)
            missing += 1
            continue

        iu = pid2idx.get(u)
        iv = pid2idx.get(v)
        if iu is None or iv is None:
            # protein 表里找不到这个 id
            scores.append(0.0)
            missing += 1
            continue

        s = float(cosine_similarity(X[iu], X[iv])[0, 0])
        scores.append(s)

    ppi[OUT_SCORE_COL] = scores
    ppi[OUT_PRED_COL] = (ppi[OUT_SCORE_COL] >= THRESHOLD).astype(int)

    # ---- E. 输出
    out_path = os.path.splitext(PPI_CSV)[0] + "_with_ml.csv"
    ppi.to_csv(out_path, index=False)

    print("==== Done ====")
    print(f"Proteins with biogrid_id: {len(prot_ids)}")
    print(f"Edges: {len(ppi)}")
    print(f"Missing endpoint (id not found / invalid): {missing}")
    print(f"{OUT_SCORE_COL}: min={ppi[OUT_SCORE_COL].min():.4f}, mean={ppi[OUT_SCORE_COL].mean():.4f}, max={ppi[OUT_SCORE_COL].max():.4f}")
    print(f"{OUT_PRED_COL} positive rate: {ppi[OUT_PRED_COL].mean():.4f}")
    print(f"Saved: {out_path}")


if __name__ == "__main__":
    main()


Scoring edges: 100%|██████████| 692420/692420 [03:35<00:00, 3218.29it/s]


==== Done ====
Proteins with biogrid_id: 19779
Edges: 692420
Missing endpoint (id not found / invalid): 0
ml_score: min=0.0000, mean=0.0513, max=1.0000
ml_pred positive rate: 0.0203
Saved: /home/yyyy/codework/GARplus/DiGress/DiGress-main/data/PPI/raw/protein_protein_with_type_with_ml.csv


# 根据数值分布选择阈值

In [5]:
import pandas as pd
df = pd.read_csv("/home/yyyy/codework/GARplus/DiGress/DiGress-main/data/PPI/raw/protein_protein_with_type_with_ml.csv")
print(df["ml_score"].quantile([0, .5, .9, .95, .98, .99, .995, 1]))


0.000    0.000000
0.500    0.019397
0.900    0.087725
0.950    0.183621
0.980    0.503767
0.990    0.751938
0.995    0.972701
1.000    1.000000
Name: ml_score, dtype: float64


In [8]:
PPI_CSV = "/home/yyyy/codework/GARplus/DiGress/DiGress-main/data/PPI/raw/protein_protein_with_type.csv"
import pandas as pd
df = pd.read_csv(PPI_CSV)
print(df.columns.tolist())


['#BioGRID Interaction ID', 'Entrez Gene Interactor A', 'Entrez Gene Interactor B', 'BioGRID ID Interactor A', 'BioGRID ID Interactor B', 'Systematic Name Interactor A', 'Systematic Name Interactor B', 'Official Symbol Interactor A', 'Official Symbol Interactor B', 'Synonyms Interactor A', 'Synonyms Interactor B', 'Experimental System', 'Experimental System Type', 'Author', 'Publication Source', 'Organism ID Interactor A', 'Organism ID Interactor B', 'Throughput', 'Score', 'Modification', 'Qualifications', 'SWISS-PROT Accessions Interactor A', 'TREMBL Accessions Interactor A', 'REFSEQ Accessions Interactor A', 'SWISS-PROT Accessions Interactor B', 'TREMBL Accessions Interactor B', 'REFSEQ Accessions Interactor B', 'Ontology Term IDs', 'Ontology Term Names', 'Ontology Term Categories', 'Ontology Term Qualifier IDs', 'Ontology Term Qualifier Names', 'Ontology Term Types', 'Organism Name Interactor A', 'Organism Name Interactor B', 'index_A', 'index_B', 'edge_semantic', 'edge_reason', 'ty